# Peek at KO/EC links — annotation rows ↔ biosamples / studies

Worked-example queries showing how to navigate from a row in
`nmdc_results.annotation_kegg_orthology` (or `annotation_enzyme_commission`)
out to the biosample, study, and file metadata in `nmdc_metadata`.

**Anchor columns on the annotation tables:**

| column | meaning | join target |
|---|---|---|
| `workflow_run_id` | `nmdc:wfmgan-...` MetagenomeAnnotation activity | `nmdc_metadata.workflow_execution_set.id` |
| `data_object_id`  | `nmdc:dobj-...` source TSV file | `nmdc_metadata.data_object_set.id` |
| `gene_id`         | `<workflow_run_id>_<contig>_<start>_<end>` | local; matches functional GFFs (issues #84, #86–#90) |
| `ncbi_taxid`      | NCBI taxon for the BLAST hit | external (no native BERDL taxonomy table) |
| `annotation_id`   | `KO:Kxxxxx` or `EC:n.n.n.n` | KEGG / EC ontologies |

**The chain:**

```
annotation_*.workflow_run_id
  → workflow_execution_set.id  (.was_informed_by is array of dgns IDs)
  → data_generation_set.id
    → data_generation_set_has_input.has_input            (biosample IDs)
    → data_generation_set_associated_studies.associated_studies
  → biosample_set.{env_*, geo_loc_name_*, depth, …}
```

### Spark Connect

In [ ]:
spark = get_spark_session(app_name="peek_ko_ec_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

### Preflight: confirm the annotation tables are registered

In [ ]:
for tbl in ("annotation_kegg_orthology", "annotation_enzyme_commission"):
    if spark.sql(f"SHOW TABLES IN nmdc_results LIKE '{tbl}'").count() == 0:
        raise RuntimeError(f"nmdc_results.{tbl} not registered — run ingest_ko_ec_annotations.ipynb first")
    print(f"OK: nmdc_results.{tbl}")

### 1. Pick an anchor row

Grab one annotation row to drive the rest of the joins. Same recipe works for EC.

In [ ]:
anchor = (
    spark.sql("""
        SELECT gene_id, annotation_id, workflow_run_id, data_object_id, ncbi_taxid
        FROM nmdc_results.annotation_kegg_orthology
        LIMIT 1
    """)
    .toPandas()
    .iloc[0]
)
WORKFLOW_RUN_ID = anchor["workflow_run_id"]
DATA_OBJECT_ID  = anchor["data_object_id"]
ANNOTATION_ID   = anchor["annotation_id"]
print(anchor.to_string())

### 2. `workflow_run_id` → workflow_execution_set

`was_informed_by` is **array-typed** (a workflow run can be informed by
multiple data generations). Use `array_contains` or `EXPLODE` to join through
it — equality won't work.

In [ ]:
spark.sql(f"""
    SELECT id, type, was_informed_by, started_at_time, ended_at_time
    FROM nmdc_metadata.workflow_execution_set
    WHERE id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 3. workflow_execution → data_generation_set (the sequencing run)

In [ ]:
spark.sql(f"""
    SELECT dg.id, dg.type, dg.name, dg.processing_institution,
           dg.principal_investigator_name, dg.start_date, dg.end_date
    FROM nmdc_metadata.workflow_execution_set we
    LATERAL VIEW EXPLODE(we.was_informed_by) info AS dgns_id
    JOIN nmdc_metadata.data_generation_set dg
      ON dg.id = info.dgns_id
    WHERE we.id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 4. data_generation → biosample (via the `_has_input` link table)

Multi-valued slots in MongoDB are flattened to dedicated link tables in the
lakehouse: `<parent>_<slot>` with `parent_id` + the slot value. Same shape
for `_associated_studies`, `_has_output`, etc.

In [ ]:
spark.sql(f"""
    SELECT bs.id AS biosample_id,
           bs.name,
           bs.env_broad_scale_term_id, bs.env_local_scale_term_id, bs.env_medium_term_id,
           bs.geo_loc_name_has_raw_value, bs.depth_has_numeric_value, bs.depth_has_unit
    FROM nmdc_metadata.workflow_execution_set we
    LATERAL VIEW EXPLODE(we.was_informed_by) info AS dgns_id
    JOIN nmdc_metadata.data_generation_set_has_input dhi
      ON dhi.parent_id = info.dgns_id
    JOIN nmdc_metadata.biosample_set bs
      ON bs.id = dhi.has_input
    WHERE we.id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 5. data_generation → study

In [ ]:
spark.sql(f"""
    SELECT s.id AS study_id, s.name, s.title, s.principal_investigator_name
    FROM nmdc_metadata.workflow_execution_set we
    LATERAL VIEW EXPLODE(we.was_informed_by) info AS dgns_id
    JOIN nmdc_metadata.data_generation_set_associated_studies dgs
      ON dgs.parent_id = info.dgns_id
    JOIN nmdc_metadata.study_set s
      ON s.id = dgs.associated_studies
    WHERE we.id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 6. `data_object_id` → data_object_set (file-level provenance)

Useful for verifying which source TSV a row came from, file size, MD5,
URL, etc. Not needed for downstream science.

In [ ]:
spark.sql(f"""
    SELECT id, name, data_object_type, file_size_bytes, md5_checksum, url
    FROM nmdc_metadata.data_object_set
    WHERE id = '{DATA_OBJECT_ID}'
""").toPandas()

### 7. Cross-check against `functional_annotation_agg`

`functional_annotation_agg` is a precomputed `(workflow_run, gene_function_id, count)`
rollup. Two caveats when joining to it:

1. **EC is absent** — `functional_annotation_agg` only carries `PFAM`,
   `KEGG.ORTHOLOGY`, and `COG`. The new `annotation_enzyme_commission` table
   is the only source of EC in BERDL.
2. **KO prefix differs** — `KO:Kxxxxx` here vs `KEGG.ORTHOLOGY:Kxxxxx` there.
   Translate before joining. The two values per workflow run *should* line up
   (`COUNT(*)` in `annotation_kegg_orthology` ≈ `SUM(count)` in the agg);
   any drift is worth flagging.

In [ ]:
spark.sql(f"""
    WITH ours AS (
        SELECT 'KEGG.ORTHOLOGY:' || SUBSTRING(annotation_id, 4) AS gene_function_id,
               COUNT(*) AS hits_in_ko_table
        FROM nmdc_results.annotation_kegg_orthology
        WHERE workflow_run_id = '{WORKFLOW_RUN_ID}'
        GROUP BY 1
    ),
    theirs AS (
        SELECT gene_function_id, count AS count_in_agg
        FROM nmdc_metadata.functional_annotation_agg
        WHERE was_generated_by = '{WORKFLOW_RUN_ID}'
          AND gene_function_id LIKE 'KEGG.ORTHOLOGY:%'
    )
    SELECT COALESCE(ours.gene_function_id, theirs.gene_function_id) AS gene_function_id,
           ours.hits_in_ko_table,
           theirs.count_in_agg,
           ours.hits_in_ko_table - theirs.count_in_agg AS diff
    FROM ours FULL OUTER JOIN theirs USING (gene_function_id)
    ORDER BY ABS(COALESCE(ours.hits_in_ko_table, 0) - COALESCE(theirs.count_in_agg, 0)) DESC
    LIMIT 20
""").toPandas()

### 8. End-to-end: KO hits per biosample for a chosen KO

The query that motivated the table — "how often does this KEGG Orthology
appear in each biosample?". Filter early on `annotation_id` so the heavy
table is the smallest leg of the join.

In [ ]:
TARGET_KO = ANNOTATION_ID  # reuse the KO we anchored on; substitute any 'KO:Kxxxxx'

spark.sql(f"""
    WITH ko_hits AS (
        SELECT workflow_run_id, COUNT(*) AS n_hits
        FROM nmdc_results.annotation_kegg_orthology
        WHERE annotation_id = '{TARGET_KO}'
        GROUP BY workflow_run_id
    )
    SELECT bs.id AS biosample_id,
           bs.env_broad_scale_term_id,
           bs.env_medium_term_id,
           bs.geo_loc_name_has_raw_value,
           SUM(k.n_hits) AS total_hits
    FROM ko_hits k
    JOIN nmdc_metadata.workflow_execution_set we
      ON we.id = k.workflow_run_id
    LATERAL VIEW EXPLODE(we.was_informed_by) info AS dgns_id
    JOIN nmdc_metadata.data_generation_set_has_input dhi
      ON dhi.parent_id = info.dgns_id
    JOIN nmdc_metadata.biosample_set bs
      ON bs.id = dhi.has_input
    GROUP BY 1, 2, 3, 4
    ORDER BY total_hits DESC
    LIMIT 20
""").toPandas()